In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://www.walkerplus.com/event_list/ar0313/sc309880d/',
   'title': '東京駅(東京都)周辺のイベント - ウォーカープラス',
   'content': '## ニュース. ## おでかけ情報. ## 季節特集. ## おでかけトレンド. ## エリア情報. # 東京駅(東京都)周辺のイベント. ### 日付から探す. 十二神将立像(巳神)\u3000鎌倉時代(13世紀)\u3000静嘉堂蔵. クロード・モネ《戸外の人物習作ー日傘を持つ右向きの女》1886年、油彩・カンヴァス、オルセー美術館蔵. 歴史リアル謎解きゲーム「謎の城」in 日本橋「発明家／人斬り-平賀源内-」. 小林 清親《東京新大橋雨中図》明治9年(1876年) スミソニアン国立アジア美術館. 《平治物語絵巻 常盤巻》(部分)、 鎌倉時代13世紀、重要文化財、石橋財団アーティゾン美術館. 2025年11月21日(金)～2026年2月28日(土) (予定). アビー×リトルツインスターズ POP-UP STORE(ポップアップストア). ## おすすめ記事. ## 閲覧履歴. ## よく使われる検索条件. ## イベントランキング. ## テーマWalker. ## 季節特集. ## おでかけ特集. ## おでかけトレンド. ## 編集部イチオシ.',
   'score': 0.99994123,
   'raw_content': None},
  {'url': 'https://ekitan.com/event/station-2590',
   'title': '東京駅周辺のイベント - 駅探',
   'content': '## 東京駅のイベント一覧. ### 条件指定. * #### 日本の桜名所100選\u3000赤城南面千本桜！菜の花・桜・花桃の中を駆け抜けるトロッコわたらせ渓谷号. + 二重橋前駅／東京駅／大手町駅(東京)駅. 期間2026年2月17日(火)～2月18日(水) このイベントは終了しました. * #### 平日昼間\u3000初心者向け 山手線1周ラン 約10～42キロ（キロ約7～8分）. + 東京駅／二重橋前駅／京橋駅(東京)駅. 期間2026年2月6

In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県を比較すると、沖縄県の方が広いです。

- **東京都**の面積は約2,194平方キロメートルです。
- **沖縄県**の面積は約2,271平方キロメートルです。

したがって、沖縄県は東京都よりも約77平方キロメートル広いです。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内の東京駅周辺のイベントに関する情報をいくつかご紹介します。

1. **50代・60代・クリスマス特別・笑顔あふれる食事会**
   - 【日時】現在予約受付中
   - 【場所】東京都中央区日本橋2-3-18
   - 昼食会やドライブ、国内外の旅行など、さまざまなアクティビティが用意されています。詳細は[こちら](https://www.ya-7.com/event_info.php?EventNum=24461&ya7_domain_name=ya7)で。

2. **行幸地下ギャラリー | 丸の内スぺースアンドメディア**
   - 【内容】三階にあるアートスペースでの展示が行なわれています。現代美術に関する展示やイベントも進行中。
   - 詳細は[こちら](https://www.marunouchispacemedia.jp/e/spaces/5)で。

3. **MARUNOUCHI STREET PARK**
   - 【開催期間】2025年11月13日から12月25日まで
   - 【開場時間】11:00〜22:00
   - 冬に開催されるこの公園で、さまざまな音楽、ランチエリア、アート展示が行われる予定です。詳細は[こちら](https://marunouchi-streetpark.com/)で。

4. **ポップアップショップ（Star Wars, Coji-Coji）**
   - COJI-COJI THE NONSENSE WORLD POP UP SHOPとSTAR WARS POP UP STOREが、2026年2月20日から3月5日まで開催されます。
   - 詳細は[こちら](https://www.tokyoeki-1bangai.co.jp/event/)で。

以上が、最近1ヶ月以内に東京駅周辺で開催されるイベントの一部です。興味のあるイベントについてぜひ足を運んでみてください！


In [ ]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:おはよう'

おはようございます！今日はどんなことをお手伝いできますか？


'質問:何してる'

私はあなたの質問に答えたり、情報を提供したりするためにここにいます。何か特定のことが知りたいですか？

---ご利用ありがとうございました！---
